# backwardCGM-PD — Ablation: hai backend khớp mô hình RCONThực nghiệm **mở rộng** do nhóm bổ sung, không có trong bài báo gốc.Bản tái thực nghiệm chạy các cấu hình nhỏ (`p = 8, 12`) bằng backend `grc_ipms`,tái tạo đúng cách mã R gốc khớp mô hình ứng viên, và các cấu hình lớn(`p = 16, 20`) bằng backend `mle` nhanh hơn. Vì chưa cấu hình nào từng chạy cảhai cách trên cùng một máy, thời gian giữa hai nhóm không so sánh trực tiếpđược — báo cáo có nêu rõ hạn chế này.Notebook này lấp khoảng trống đó. Với mỗi cấu hình, thuật toán chạy **hai lầntrên cùng dữ liệu, cùng số replicate, cùng số worker, cùng phần cứng**, chỉ đổibackend. Nhờ vậy tách được ảnh hưởng của thủ tục khớp khỏi mọi yếu tố khác.Hai câu hỏi được trả lời:1. `mle` nhanh hơn `grc_ipms` bao nhiêu lần?2. Hai backend có chọn ra cùng mô hình không? Nếu khác nhau thì việc so sánh   thời gian trong báo cáo sẽ dựa trên hai thủ tục không làm cùng một việc.Hãy **Add Input** dataset `backwardCGM-PD`.

In [ ]:
import importlib.util, subprocess, sys
required = {"rdata": "rdata>=0.11", "networkx": "networkx>=3.0", "joblib": "joblib>=1.3"}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies: OK")

In [ ]:
from pathlib import Path
import json, shutil, zipfile
import pandas as pd
from IPython.display import display

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/backwardCGM-PD")
RESULTS = Path("/kaggle/working/ablation-results")
RESULTS.mkdir(parents=True, exist_ok=True)

checkpoint_archives = list(INPUT_ROOT.rglob("ablation-results.zip"))
if checkpoint_archives:
    with zipfile.ZipFile(checkpoint_archives[0]) as archive:
        archive.extractall(RESULTS)
    print("Restored checkpoint:", checkpoint_archives[0])

archives = list(INPUT_ROOT.rglob("backwardCGM-PD-kaggle-dataset.zip"))
scripts = list(INPUT_ROOT.rglob("python-port/experiments/simulation.py"))
if archives:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(WORK_ROOT)
elif scripts:
    shutil.copytree(scripts[0].parents[2], WORK_ROOT, dirs_exist_ok=True)
else:
    raise FileNotFoundError("Hãy Add Input dataset backwardCGM-PD")

PORT_ROOT = WORK_ROOT / "python-port"
print("Dataset: OK\nOutput:", RESULTS)

In [ ]:
# Thực nghiệm mở rộng do nhóm viết, chưa có trong dataset gốc nên nhúng thẳng vào
# notebook. Mã giống hệt tệp cùng tên trong github.com/nhantrnh/DataMining.
SCRIPT = PORT_ROOT / "experiments" / "ablation_backend.py"
SCRIPT.write_text('"""Head-to-head comparison of the two RCON fitting backends.\n\nThe reproduction runs the small configurations (``p = 8, 12``) with the\n``grc_ipms`` backend, which mirrors how the original R code fits candidate\nmodels, and the large ones (``p = 16, 20``) with the faster ``mle`` backend.\nBecause no configuration was ever run both ways on the same machine, the\nruntimes reported for the two groups are not directly comparable, and the\nreport says so explicitly.\n\nThis script closes that gap. For each configuration it runs the identical\nsearch twice - same data, same replicates, same worker count, same machine -\nchanging only the backend. That isolates the effect of the fitting procedure\nfrom every other factor, which is what an ablation is meant to do.\n\nTwo questions are answered:\n\n1. How much faster is ``mle`` than ``grc_ipms``?\n2. Do the two backends select the same models? If they diverge, the runtime\n   comparison in the report would rest on procedures that are not doing the\n   same thing.\n\nUsage::\n\n    python experiments/ablation_backend.py --scenario A --p 8 \\\n        --replicates 10 --output results/ablation-A-p8.json\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nREPO_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(REPO_ROOT / "src"))\n\nfrom backward_cgm_pd.article_graphs import article_scenario_graph\nfrom backward_cgm_pd.graph import graph_key\nfrom backward_cgm_pd.io import load_simulated_datasets, write_json\nfrom backward_cgm_pd.metrics import average_metrics, recovery_metrics\nfrom backward_cgm_pd.search_submodel import backward_submodel\nfrom backward_cgm_pd.search_tau import backward_cgm_pd\n\nSCENARIO_CODE = {"A": "11", "B": "22"}\nSEARCHES = {"tau": backward_cgm_pd, "submodel": backward_submodel}\nBACKENDS = ("grc_ipms", "mle")\n\n\ndef saved_data_path(scenario: str, p: str) -> Path:\n    return (\n        REPO_ROOT\n        / "simulation"\n        / "simulated-data"\n        / f"simdf_{SCENARIO_CODE[scenario]}_{p}.RData"\n    )\n\n\ndef run_ablation(\n    scenario: str,\n    p: str,\n    *,\n    method: str,\n    replicates: int,\n    alpha: float,\n    itmax: int,\n    parallel: int,\n    verbose: bool,\n) -> dict[str, object]:\n    datasets = load_simulated_datasets(saved_data_path(scenario, p))[:replicates]\n    truth = article_scenario_graph(scenario, int(p))\n    search = SEARCHES[method]\n\n    rows: list[dict[str, object]] = []\n    for index, data in enumerate(datasets, 1):\n        selected: dict[str, object] = {}\n        for backend in BACKENDS:\n            started = time.perf_counter()\n            result = search(\n                data,\n                alpha=alpha,\n                itmax=itmax,\n                rcon_backend=backend,\n                n_jobs=parallel,\n            )\n            runtime = time.perf_counter() - started\n            selected[backend] = result.model\n            rows.append(\n                {\n                    "replicate": index,\n                    "backend": backend,\n                    "runtime_seconds": runtime,\n                    "iterations": result.iterations,\n                    "number_models": result.number_models,\n                    "pvalue": result.pvalue,\n                    "number_edges": len(result.model.E),\n                    "model": result.model,\n                    "metrics": recovery_metrics(result.model, truth).to_dict(),\n                }\n            )\n            if verbose:\n                print(\n                    f"[{scenario}/p={p}] replicate {index}/{len(datasets)} "\n                    f"{backend}: {runtime:.2f}s |E|={len(result.model.E)}",\n                    flush=True,\n                )\n        same = graph_key(selected["grc_ipms"]) == graph_key(selected["mle"])\n        rows[-1]["same_model_as_other_backend"] = same\n        rows[-2]["same_model_as_other_backend"] = same\n\n    summary: list[dict[str, object]] = []\n    for backend in BACKENDS:\n        subset = [row for row in rows if row["backend"] == backend]\n        runtimes = [float(row["runtime_seconds"]) for row in subset]\n        summary.append(\n            {\n                "scenario": scenario,\n                "p": int(p),\n                "method": method,\n                "backend": backend,\n                "replicates": len(subset),\n                "mean_runtime": float(np.mean(runtimes)),\n                "sd_runtime": float(np.std(runtimes, ddof=1)) if len(runtimes) > 1 else 0.0,\n                "mean_iterations": float(np.mean([r["iterations"] for r in subset])),\n                "mean_number_models": float(np.mean([r["number_models"] for r in subset])),\n                "mean_number_edges": float(np.mean([r["number_edges"] for r in subset])),\n                **average_metrics(\n                    [recovery_metrics(r["model"], truth) for r in subset]\n                ),\n            }\n        )\n\n    ipms_time = summary[0]["mean_runtime"]\n    mle_time = summary[1]["mean_runtime"]\n    agreement = [row for row in rows if row["backend"] == "mle"]\n    n_same = sum(1 for row in agreement if row.get("same_model_as_other_backend"))\n\n    return {\n        "settings": {\n            "scenario": scenario,\n            "p": int(p),\n            "method": method,\n            "replicates": len(datasets),\n            "alpha": alpha,\n            "itmax": itmax,\n            "parallel": parallel,\n        },\n        "truth": truth,\n        "speedup_mle_over_ipms": float(ipms_time / mle_time) if mle_time else None,\n        "identical_models": n_same,\n        "identical_model_rate": n_same / len(agreement) if agreement else None,\n        "runs": rows,\n        "summary": summary,\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--scenario", choices=["A", "B"], required=True)\n    parser.add_argument("--p", choices=["8", "12", "16", "20"], required=True)\n    parser.add_argument("--method", choices=list(SEARCHES), default="tau")\n    parser.add_argument("--replicates", type=int, default=10)\n    parser.add_argument("--alpha", type=float, default=0.05)\n    parser.add_argument("--itmax", type=int, default=500)\n    parser.add_argument("--parallel", type=int, default=3)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--verbose", action="store_true")\n    args = parser.parse_args()\n\n    result = run_ablation(\n        args.scenario,\n        args.p,\n        method=args.method,\n        replicates=args.replicates,\n        alpha=args.alpha,\n        itmax=args.itmax,\n        parallel=args.parallel,\n        verbose=args.verbose,\n    )\n\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    write_json(result, args.output)\n    frame = pd.DataFrame(result["summary"])\n    frame.to_csv(args.output.with_name(f"{args.output.stem}-summary.csv"), index=False)\n\n    print(f"Wrote {args.output}")\n    print(frame.to_string(index=False))\n    print(\n        f"\\nspeedup (ipms/mle) = {result[\'speedup_mle_over_ipms\']:.2f}x  |  "\n        f"mô hình trùng nhau: {result[\'identical_models\']}/"\n        f"{result[\'settings\'][\'replicates\']}"\n    )\n\n\nif __name__ == "__main__":\n    main()\n')

data_dir = PORT_ROOT / "simulation"
data_dir.mkdir(exist_ok=True)
target = data_dir / "simulated-data"
if not target.exists():
    target.symlink_to(WORK_ROOT / "simulation/simulated-data")
assert (target / "simdf_11_8.RData").exists(), "Liên kết dữ liệu không đúng"
print("Đã ghi", SCRIPT)

## Cấu hình`grc_ipms` chậm hơn nhiều nên notebook mặc định chạy `p = 8, 12` với 10replicate. Bật `INCLUDE_P16` nếu session đủ dài.

In [ ]:
REPLICATES = 10
PARALLEL = 3          # giữ đúng số worker mà mã R gốc dùng
METHOD = "tau"
INCLUDE_P16 = False

ps = ["8", "12"] + (["16"] if INCLUDE_P16 else [])
print({"replicates": REPLICATES, "parallel": PARALLEL, "method": METHOD, "p": ps})

In [ ]:
import subprocess, sys, time

started = time.perf_counter()
for scenario in ("A", "B"):
    for p in ps:
        output = RESULTS / f"ablation-{scenario}-p{p}.json"
        if output.exists():
            print(f"Bỏ qua {scenario}/p={p}: đã có kết quả")
            continue
        command = [
            sys.executable, "-u", str(PORT_ROOT / "experiments/ablation_backend.py"),
            "--scenario", scenario, "--p", p, "--method", METHOD,
            "--replicates", str(REPLICATES), "--parallel", str(PARALLEL),
            "--output", str(output),
        ]
        print("Running:", " ".join(command), flush=True)
        subprocess.run(command, cwd=PORT_ROOT, check=True)
print(f"\nTổng thời gian: {time.perf_counter() - started:.1f}s")

## Tổng hợp

In [ ]:
rows = []
for path in sorted(RESULTS.glob("ablation-*.json")):
    d = json.loads(path.read_text())
    s = d["settings"]
    ipms, mle = d["summary"]
    rows.append({
        "scenario": s["scenario"],
        "p": s["p"],
        "runtime_ipms": ipms["mean_runtime"],
        "runtime_mle": mle["mean_runtime"],
        "speedup": d["speedup_mle_over_ipms"],
        "identical_models": f"{d['identical_models']}/{s['replicates']}",
        "ePPV_ipms": ipms["ePPV"],
        "ePPV_mle": mle["ePPV"],
        "eTPR_ipms": ipms["eTPR"],
        "eTPR_mle": mle["eTPR"],
    })
summary = pd.DataFrame(rows).sort_values(["scenario", "p"])
summary.to_csv(RESULTS / "ablation-all-configurations.csv", index=False)
display(summary)

archive = shutil.make_archive("/kaggle/working/ablation-results", "zip", root_dir=RESULTS)
print("Download:", archive)

**Đọc kết quả:** cột `speedup` là tỉ số thời gian trung bình `grc_ipms` chia cho`mle`. Cột `identical_models` cho biết trong bao nhiêu replicate hai backend chọnra đúng cùng một mô hình, kể cả các lớp màu.Nếu `identical_models` đạt tối đa và các cột độ đo phục hồi trùng nhau, kết luậnlà hai backend tương đương về mặt thống kê và chỉ khác nhau ở chi phí tính toán —khi đó việc dùng `mle` cho các cấu hình lớn là hợp lệ, không làm thay đổi kết quảkhoa học.